In [ ]:
from pathlib import Path
import re
import yaml
import os
from openpyxl import load_workbook

In [ ]:
!date
jupyter_dir = os.getcwd()
print(f"DIR: {jupyter_dir}")

In [ ]:
#!date
#excel_file = Path("./Subscriber.xlsx")
#out_yaml = Path("./output2/output.yaml")

In [ ]:
!date
from pathlib import Path
import re
import yaml
from openpyxl import load_workbook

# ====== Excel layout ======
SHEETS = ["SourceAddress", "BlackList", "WhiteList", "CategoryList"]

ROW_IDENTIFIER = 4
ROW_DATA_START = 5
COL_START = 3

# ====== BIG-IP fixed rules ======
MASK_FIXED = "255.255.255.255"
VS_RULE = "/Common/Shared/url_filter_irule"
TMC_DESTINATION_ADDRESS_INLINE = "192.168.23.12"

TCP_PROFILES = [
    "/Common/Shared/{id}_Profile",
    "/Common/Shared/{id}_stats",
    "/Common/tcp",
]

UDP_PROFILES = [
    "/Common/Shared/{id}_Profile",
    "/Common/Shared/{id}_stats",
    "/Common/Shared/udp_gtm_dns",
]

split_pattern = re.compile(r"[,\n\r]+")


def split_values(v):
    if v is None:
        return []
    s = str(v).strip()
    if not s:
        return []
    return [x.strip() for x in split_pattern.split(s) if x.strip()]


def read_identifier_columns(ws):
    col_to_id = {}
    for col in range(COL_START, ws.max_column + 1):
        v = ws.cell(row=ROW_IDENTIFIER, column=col).value
        if v is None:
            continue
        ident = str(v).strip()
        if ident:
            col_to_id[col] = ident
    return col_to_id


def read_sheet_values_by_identifier(ws):
    col_to_id = read_identifier_columns(ws)
    result = {ident: [] for ident in col_to_id.values()}

    for col, ident in col_to_id.items():
        values = []
        for row in range(ROW_DATA_START, ws.max_row + 1):
            cell_v = ws.cell(row=row, column=col).value
            values.extend(split_values(cell_v))

        values = [v for v in values if v and v.strip()]
        result[ident] = sorted(set(values))

    return result


def build_virtual_servers_expected(identifier: str):
    return {
        "TCP": {
            "name": f"/Common/Shared/{identifier}-VIP_P_TCP",
            "ip_protocol": "tcp",
            "mask": MASK_FIXED,
            "profiles": sorted({p.format(id=identifier) for p in TCP_PROFILES}),
            "rules": [VS_RULE],
            "traffic_matching_criteria": f"/Common/Shared/{identifier}-VIP_P_TCP_VS_TMC_OBJ",
        },
        "UDP": {
            "name": f"/Common/Shared/{identifier}-VIP_P_UDP",
            "ip_protocol": "udp",
            "mask": MASK_FIXED,
            "profiles": sorted({p.format(id=identifier) for p in UDP_PROFILES}),
            "rules": [VS_RULE],
            "traffic_matching_criteria": f"/Common/Shared/{identifier}-VIP_P_UDP_VS_TMC_OBJ",
        },
    }


def build_traffic_matching_criteria_expected(identifier: str):
    return {
        "TCP": {
            "name": f"/Common/Shared/{identifier}-VIP_P_TCP_VS_TMC_OBJ",
            "destination_address_inline": TMC_DESTINATION_ADDRESS_INLINE,
            "source_address_list": f"/Common/Shared/{identifier}_addresslist",
        },
        "UDP": {
            "name": f"/Common/Shared/{identifier}-VIP_P_UDP_VS_TMC_OBJ",
            "destination_address_inline": TMC_DESTINATION_ADDRESS_INLINE,
            "source_address_list": f"/Common/Shared/{identifier}_addresslist",
        },
    }


def excel_to_expected(excel_path: str | Path):
    excel_path = Path(excel_path)
    wb = load_workbook(excel_path, data_only=True)

    sheet_data = {}
    for sheet in SHEETS:
        if sheet not in wb.sheetnames:
            raise ValueError(f"Sheet not found: {sheet}. Available: {wb.sheetnames}")
        ws = wb[sheet]
        sheet_data[sheet] = read_sheet_values_by_identifier(ws)

    identifiers = set()
    for d in sheet_data.values():
        identifiers |= set(d.keys())

    expected = {}
    for ident in sorted(identifiers):
        expected[ident] = {

            # ここを変更
            "source_address_list": sheet_data["SourceAddress"].get(ident, []),

            "data_groups": {
                "blacklist": sheet_data["BlackList"].get(ident, []),
                "whitelist": sheet_data["WhiteList"].get(ident, []),
                "category": sheet_data["CategoryList"].get(ident, []),
            },

            "virtual_servers": build_virtual_servers_expected(ident),

            "traffic_matching_criteria": build_traffic_matching_criteria_expected(ident),
        }

    return expected


# ====== Run ======
excel_file = Path("./Subscriber.xlsx")
out_yaml = Path("./output2/output.yaml")

expected = excel_to_expected(excel_file)

out_yaml.parent.mkdir(parents=True, exist_ok=True)

with open(out_yaml, "w", encoding="utf-8") as f:
    yaml.dump(expected, f, allow_unicode=True, sort_keys=False)

print(f"Exported: {out_yaml} (identifiers={len(expected)})")


In [ ]:
#expected = excel_to_expected(excel_file)
#
## 出力ディレクトリが無ければ作成
#out_yaml.parent.mkdir(parents=True, exist_ok=True)
#
## YAML出力
#with open(out_yaml, "w", encoding="utf-8") as f:
#    yaml.dump(expected, f, allow_unicode=True, sort_keys=False)
#
#print(f"Exported: {out_yaml} (identifiers={len(expected)})")

# actual

In [ ]:
from pathlib import Path
import yaml
import re

# ====== Input / Output ======
INPUT_VS = Path("./input/tmsh-vs.txt")
INPUT_DG = Path("./input/tmsh-dg.txt")
INPUT_TMC = Path("./input/tmsh-tmc.txt")
INPUT_AL = Path("./input/tmsh-al.txt")

OUTPUT_YAML = Path("./output2/actual.yaml")


# ====== Utilities ======
def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8")

def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

def sorted_unique(values):
    return sorted(set(values))

def normalize_shared_object(value: str) -> str:
    """
    tmsh short name -> canonical path
    """
    value = value.strip()

    if value.startswith("/Common/"):
        return value

    if value in ("tcp", "udp"):
        return f"/Common/{value}"

    return f"/Common/Shared/{value}"

def ensure_identifier_root(actual: dict, identifier: str):
    if identifier not in actual:
        actual[identifier] = {
            "source_address_list": [],
            "data_groups": {
                "blacklist": [],
                "whitelist": [],
                "category": [],
            },
            "virtual_servers": {},
            "traffic_matching_criteria": {},
        }


# ====== Name extractors ======
def extract_identifier_and_proto_from_vs_name(name: str):
    m = re.match(r"^(.+)-VIP_P_(TCP|UDP)$", name)
    if not m:
        return None, None
    return m.group(1), m.group(2)

def extract_identifier_and_kind_from_dg_name(name: str):
    m = re.match(r"^(.+)_(blacklist|category|whitelist)$", name)
    if not m:
        return None, None
    return m.group(1), m.group(2)

def extract_identifier_and_proto_from_tmc_name(name: str):
    m = re.match(r"^(.+)-VIP_P_(TCP|UDP)_VS_TMC_OBJ$", name)
    if not m:
        return None, None
    return m.group(1), m.group(2)

def extract_identifier_from_address_list_name(name: str):
    m = re.match(r"^(.+)_addresslist$", name)
    if not m:
        return None
    return m.group(1)


# ====== Block splitter ======
def split_tmsh_blocks(text: str, object_prefix: str):
    lines = text.splitlines()
    blocks = []

    in_block = False
    current_name = None
    current_lines = []
    brace_depth = 0

    for line in lines:
        stripped = line.strip()

        if not in_block and line.startswith(object_prefix):
            in_block = True
            current_name = line[len(object_prefix):].strip()

            # 末尾の { を除去
            if current_name.endswith("{"):
                current_name = current_name[:-1].strip()

            current_lines = []
            brace_depth = 1   # オブジェクト開始の { 分
            continue

        if in_block:
            # 現在行を保存
            current_lines.append(line)

            # 行内の { } 数を数える
            brace_depth += line.count("{")
            brace_depth -= line.count("}")

            # 0 になったらブロック終了
            if brace_depth == 0:
                # 最後の "}" 行は body から外す
                if current_lines and current_lines[-1].strip() == "}":
                    current_lines = current_lines[:-1]

                blocks.append((current_name, "\n".join(current_lines)))

                in_block = False
                current_name = None
                current_lines = []
                brace_depth = 0

    return blocks


# ====== Parsers ======
def parse_virtual_servers(text: str) -> dict:
    actual = {}
    blocks = split_tmsh_blocks(text, "ltm virtual ")

    for raw_name, body in blocks:
        identifier, proto = extract_identifier_and_proto_from_vs_name(raw_name)
        if not identifier or not proto:
            continue

        ensure_identifier_root(actual, identifier)

        ip_protocol = None
        profiles = []
        rules = []
        traffic_matching_criteria = None

        in_profiles = False
        in_rules = False

        for line in body.splitlines():
            s = line.strip()

            if s.startswith("ip-protocol "):
                ip_protocol = s.split(maxsplit=1)[1].strip()
                continue

            if s == "profiles {":
                in_profiles = True
                continue
            if in_profiles and s == "}":
                in_profiles = False
                continue

            if s == "rules {":
                in_rules = True
                continue
            if in_rules and s == "}":
                in_rules = False
                continue

            if in_profiles:
                m = re.match(r"^(\S+)\s+\{\s*\}$", s)
                if m:
                    profiles.append(normalize_shared_object(m.group(1)))
                continue

            if in_rules:
                if s:
                    rules.append(normalize_shared_object(s))
                continue

            if s.startswith("traffic-matching-criteria "):
                raw_tmc = s.split(maxsplit=1)[1].strip()
                traffic_matching_criteria = normalize_shared_object(raw_tmc)

        actual[identifier]["virtual_servers"][proto] = {
            "name": normalize_shared_object(raw_name),
            "ip_protocol": ip_protocol,
            "profiles": sorted_unique(profiles),
            "rules": sorted_unique(rules),
            "traffic_matching_criteria": traffic_matching_criteria,
        }

    return actual


def parse_data_groups(text: str) -> dict:
    actual = {}
    blocks = split_tmsh_blocks(text, "ltm data-group internal ")

    for raw_name, body in blocks:
        identifier, kind = extract_identifier_and_kind_from_dg_name(raw_name)
        if not identifier or not kind:
            continue

        ensure_identifier_root(actual, identifier)

        records = []
        in_records = False

        for line in body.splitlines():
            s = line.strip()

            if s == "records {":
                in_records = True
                continue
            if in_records and s == "}":
                in_records = False
                continue

            if in_records:
                m = re.match(r'^"(.+?)"\s+\{\s*\}$', s)
                if m:
                    records.append(m.group(1))

        actual[identifier]["data_groups"][kind] = sorted_unique(records)

    return actual


def parse_tmc(text: str) -> dict:
    actual = {}
    blocks = split_tmsh_blocks(text, "ltm traffic-matching-criteria ")

    for raw_name, body in blocks:
        identifier, proto = extract_identifier_and_proto_from_tmc_name(raw_name)
        if not identifier or not proto:
            continue

        ensure_identifier_root(actual, identifier)

        destination_address_inline = None
        source_address_list = None

        for line in body.splitlines():
            s = line.strip()

            if s.startswith("destination-address-inline "):
                destination_address_inline = s.split(maxsplit=1)[1].strip()

            elif s.startswith("source-address-list "):
                raw_source_list = s.split(maxsplit=1)[1].strip()
                # ここだけ /Common/Shared/... の可能性もあるので normalize_shared_object で吸収
                if raw_source_list.startswith("/Common/"):
                    source_address_list = raw_source_list
                else:
                    source_address_list = normalize_shared_object(raw_source_list)

        actual[identifier]["traffic_matching_criteria"][proto] = {
            "name": normalize_shared_object(raw_name),
            "destination_address_inline": destination_address_inline,
            "source_address_list": source_address_list,
        }

    return actual


def parse_address_lists(text: str) -> dict:
    actual = {}
    blocks = split_tmsh_blocks(text, "net address-list ")

    for raw_name, body in blocks:
        identifier = extract_identifier_from_address_list_name(raw_name)
        if not identifier:
            continue

        ensure_identifier_root(actual, identifier)

        records = []
        in_addresses = False

        for line in body.splitlines():
            s = line.strip()

            if s in ("address {", "addresses {"):
                in_addresses = True
                continue
            if in_addresses and s == "}":
                in_addresses = False
                continue

            if in_addresses:
                m = re.match(r"^(\S+)\s+\{\s*\}$", s)
                if m:
                    records.append(m.group(1))

        actual[identifier]["source_address_list"] = sorted_unique(records)

    return actual


# ====== Merge ======
def deep_merge_actual(base: dict, extra: dict) -> dict:
    for identifier, data in extra.items():
        ensure_identifier_root(base, identifier)

        if data.get("source_address_list"):
            base[identifier]["source_address_list"] = data["source_address_list"]

        if "data_groups" in data:
            for kind, values in data["data_groups"].items():
                base[identifier]["data_groups"][kind] = values

        if "virtual_servers" in data:
            for proto, vs_data in data["virtual_servers"].items():
                base[identifier]["virtual_servers"][proto] = vs_data

        if "traffic_matching_criteria" in data:
            for proto, tmc_data in data["traffic_matching_criteria"].items():
                base[identifier]["traffic_matching_criteria"][proto] = tmc_data

    return base


# ====== Main ======
def build_actual_from_tmsh_files(
    vs_file: Path,
    dg_file: Path,
    tmc_file: Path,
    al_file: Path,
) -> dict:
    actual = {}

    actual = deep_merge_actual(actual, parse_virtual_servers(read_text(vs_file)))
    actual = deep_merge_actual(actual, parse_data_groups(read_text(dg_file)))
    actual = deep_merge_actual(actual, parse_tmc(read_text(tmc_file)))
    actual = deep_merge_actual(actual, parse_address_lists(read_text(al_file)))

    return dict(sorted(actual.items()))


actual = build_actual_from_tmsh_files(
    vs_file=INPUT_VS,
    dg_file=INPUT_DG,
    tmc_file=INPUT_TMC,
    al_file=INPUT_AL,
)

ensure_parent(OUTPUT_YAML)
with open(OUTPUT_YAML, "w", encoding="utf-8") as f:
    yaml.dump(actual, f, allow_unicode=True, sort_keys=False)

print(f"Exported: {OUTPUT_YAML} (identifiers={len(actual)})")
print(actual)

In [ ]:
def parse_data_groups(text: str) -> dict:
    actual = {}
    blocks = split_tmsh_blocks(text, "ltm data-group internal ")

    print(f"[DEBUG] DG blocks found: {len(blocks)}")

    for raw_name, body in blocks:
        print("-----")
        print(f"[DEBUG] raw_name: {raw_name}")
        print(f"[DEBUG] body:\n{body}")

        identifier, kind = extract_identifier_and_kind_from_dg_name(raw_name)
        print(f"[DEBUG] identifier={identifier}, kind={kind}")

        if not identifier or not kind:
            print("[DEBUG] skipped: name parse failed")
            continue

        ensure_identifier_root(actual, identifier)

        records = []
        in_records = False

        for line in body.splitlines():
            s = line.strip()
            print(f"[DEBUG] line={s!r}, in_records={in_records}")

            if s == "records {":
                in_records = True
                print("[DEBUG] records block start")
                continue

            if in_records and s == "}":
                in_records = False
                print("[DEBUG] records block end")
                continue

            if in_records:
                m = re.match(r'^"(.+?)"\s+\{\s*\}$', s)
                if m:
                    records.append(m.group(1))
                    print(f"[DEBUG] record added: {m.group(1)}")

        actual[identifier]["data_groups"][kind] = sorted_unique(records)
        print(f"[DEBUG] actual[{identifier}][data_groups][{kind}] = {sorted_unique(records)}")

    return actual

In [ ]:
def parse_data_groups(text: str) -> dict:
    actual = {}
    blocks = split_tmsh_blocks(text, "ltm data-group internal ")

    for raw_name, body in blocks:
        identifier, kind = extract_identifier_and_kind_from_dg_name(raw_name)
        if not identifier or not kind:
            continue

        ensure_identifier_root(actual, identifier)

        records = []
        in_records = False

        for line in body.splitlines():
            s = line.strip()

            if re.match(r"^records\s*\{$", s):
                in_records = True
                continue

            if in_records and s == "}":
                in_records = False
                continue

            if in_records:
                # quoted / unquoted 両対応
                # 例:
                #   "Illegal or Unethical" { }
                #   Self-Harm { }
                #   Violence { }
                m = re.match(r'^(?:"(.+?)"|(.+?))\s+\{\s*\}$', s)
                if m:
                    record = m.group(1) if m.group(1) is not None else m.group(2).strip()
                    records.append(record)

        actual[identifier]["data_groups"][kind] = sorted_unique(records)

    return actual

In [ ]:
def deep_merge_actual(base: dict, extra: dict) -> dict:
    for identifier, data in extra.items():
        ensure_identifier_root(base, identifier)

        # source_address_list
        if "source_address_list" in data:
            if data["source_address_list"] is not None:
                base[identifier]["source_address_list"] = data["source_address_list"]

        # data_groups
        if "data_groups" in data:
            if "data_groups" not in base[identifier] or not isinstance(base[identifier]["data_groups"], dict):
                base[identifier]["data_groups"] = {}

            for kind in ("blacklist", "whitelist", "category"):
                if kind in data["data_groups"]:
                    base[identifier]["data_groups"][kind] = data["data_groups"][kind]

        # virtual_servers
        if "virtual_servers" in data:
            if "virtual_servers" not in base[identifier] or not isinstance(base[identifier]["virtual_servers"], dict):
                base[identifier]["virtual_servers"] = {}

            for proto, vs_data in data["virtual_servers"].items():
                base[identifier]["virtual_servers"][proto] = vs_data

        # traffic_matching_criteria
        if "traffic_matching_criteria" in data:
            if "traffic_matching_criteria" not in base[identifier] or not isinstance(base[identifier]["traffic_matching_criteria"], dict):
                base[identifier]["traffic_matching_criteria"] = {}

            for proto, tmc_data in data["traffic_matching_criteria"].items():
                base[identifier]["traffic_matching_criteria"][proto] = tmc_data

    return base

In [ ]:
def ensure_identifier_root(actual: dict, identifier: str):
    if identifier not in actual:
        actual[identifier] = {
            "source_address_list": [],
            "data_groups": {},
            "virtual_servers": {},
            "traffic_matching_criteria": {},
        }